# LLM-FE на H&M: Llama‑3.1‑8B‑Instruct, Kaggle T4

Llama создаёт Python-программы признаков; LightGBM измеряет их качество и направляет
поиск в трёх популяциях. LightGBM используется на всех этапах: контроль, search,
confirmation и итоговая оценка. Ручной LightGBM и GNN→LightGBM сравниваются по сохранённым предсказаниям
на **тех же пользователях и кандидатах**. Это адаптация исследования, не его точное воспроизведение.

**Как запустить:**
1. В Kaggle импортируйте этот `.ipynb` через **File → Import Notebook**.
2. Выберите GPU **T4** (достаточно одной; при T4×2 используется GPU0), включите Internet.
3. Создайте Kaggle Dataset из `hm_llmfe_kaggle_fulltrain_v2.zip` и добавьте его через Input.
   Поддерживаются и исходный ZIP, и автоматически распакованные Kaggle файлы.
4. В конфиге выберите `MODEL_SOURCE = 'unsloth'` (по умолчанию, без токена) или `'hf'`.
   Для `'hf'` получите доступ к [официальной Llama](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
   и добавьте `HF_TOKEN` в **Add-ons → Secrets**. Оба источника размещены на Hugging Face.
5. Выполните **Run All**. Модель скачивается автоматически; веса не входят в архив данных.
6. Скачайте итоговый `*_results.zip` из Output. В нём программы, модели деревьев, метрики и графики.

В ZIP данных находятся **те же803377 обучающих пар /50000 user-time строк /52 снимка**, что
в локальных сравнениях. Новое отрицательное семплирование не выполняется.
Ограничена только оценка: search256, confirmation512, отдельная val1024, test2048 пользователей.
Поисковые/confirmation/итоговые val строки не пересекаются. Кандидаты ID-GNN неизменны;
позитивные test-товары в пул дополнительно не добавляются.


## Условия и отличия от статьи

| Параметр | Этот ноутбук |
|---|---|
| LLM | Llama-3.1-8B-Instruct: официальный источник Meta (`hf`) или публикация Unsloth (`unsloth`) |
| Размещение на T4 | NF4 4-bit, вычисления float16; квантизация — наше отличие |
| Генерация | temperature0.8,3 программы на один prompt, всего20 программ |
| Память поиска | 3 независимые популяции; до3 демонстраций, кластеры по validation MAP, Boltzmann sampling |
| Форма признаков | Python/pandas, `modify_features(X, history)`; без старых ручных/GNN-признаков |
| Основной evaluator | LightGBM, binary objective,150 boosting rounds,31 лист; подбор через MAP@12 |
| Проверка сопоставимости | Raw/LLM LightGBM на одних парах; те же деревья проходят confirmation и финал |
| История | Только выбранная когорта клиентов, до cutoff, последние180 дней; global counts относятся к этой когорте |
| Ресурсы | 30секунд на вызов функции признаков,4GiB на отдельный evaluator,1800секунд на обучение+оценку |
| Отбор | search→confirmation; итоговые val/test открываются после фиксации программы |

В статье рассматриваются табличные classification/regression, используются другие датасеты,
а лимит30секунд относится к полной оценке. Здесь сохраняем оригинальные803377 пар, поэтому
выделяем отдельный больший лимит для полного обучения и4GiB вместо2GiB. По прямому указанию
пользователя весь отбор переведён на LightGBM; XGBoost в этом протоколе не используется.
В Kaggle нет повторного обучения GNN. Это новый протокол v3, результаты v2 не переиспользуются.
Референсные деревья использовали те же пары, но другие схемы подбора/производные seeds;
GNN дополнительно обучалась на графе. Это сравнение качества конвейеров, не изолированный
эффект способа генерации признаков. Полные старые MAP нельзя подставлять вместо MAP этой выборки.

Источники: [LLM-FE §4.3, Appendix B](https://arxiv.org/html/2503.14434v3),
[код авторов](https://github.com/nikhilsab/LLMFE),
[NF4](https://huggingface.co/docs/transformers/v4.57.1/quantization/bitsandbytes).


In [ ]:
import subprocess, sys
# Не переустанавливаем torch/CUDA из готового окружения Kaggle.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate==1.10.1', 'bitsandbytes==0.48.1',
    'lightgbm==4.6.0', 'sentencepiece==0.2.1'])


In [ ]:
from pathlib import Path
import os, json, hashlib, time, gc, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

# Оба варианта скачиваются с Hugging Face; различается издатель репозитория.
MODEL_SOURCE = 'unsloth'  # 'unsloth' без токена; 'hf' официальный Meta с доступом/HF_TOKEN
MODEL_SOURCES = {
    'hf': {'repo_id':'meta-llama/Llama-3.1-8B-Instruct', 'requires_token':True},
    'unsloth': {'repo_id':'unsloth/Meta-Llama-3.1-8B-Instruct', 'requires_token':False},
}
if MODEL_SOURCE not in MODEL_SOURCES:
    raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
MODEL_ID = MODEL_SOURCES[MODEL_SOURCE]['repo_id']

def resolve_auth(source, environ, load_secret):
    if source not in MODEL_SOURCES:
        raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
    if not MODEL_SOURCES[source]['requires_token']:
        # False запрещает также автоматическое использование старого токена из HF-кэша.
        return False
    value = environ.get('HF_TOKEN')
    if not value:
        try:
            value = load_secret()
        except Exception:
            value = None
    if not value:
        raise RuntimeError('Для hf нужны одобренный доступ Meta и Kaggle Secret HF_TOKEN; либо выберите unsloth')
    return value

# При повторе можно указать SHA из предыдущего run_config.json; пустая строка разрешит текущую ревизию.
MODEL_REVISION = ''
WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

CONFIG = {
    'protocol': 'hm_llmfe_llama31_8b_lightgbm_fulltrain_v3', 'seed':42,
    'search_backend':'lightgbm',
    'model_id':MODEL_ID, 'model_source':MODEL_SOURCE,
    'quantization':'bnb NF4 4bit/double_quant, FP16 compute',
    'temperature':0.8, 'top_p':1.0, 'top_k':0,
    'islands':3, 'programs_per_prompt':3, 'llm_samples':20, 'max_demonstrations':3,
    'max_input_tokens':6144, 'max_new_tokens':2048,
    'train_pairs':803377, 'train_user_times':50000,
    'search_users':256, 'confirmation_users':512, 'heldout_val_users':1024, 'test_users':2048,
    'cpu_threads':min(4, os.cpu_count() or 1),
    'evaluation_memory_gb':4, 'evaluation_timeout_seconds':1800,
    'feature_call_timeout_seconds':30, 'lightgbm_rounds':150,
    'paper_reproduction':False, 'candidate_protocol':'exact saved ID-GNN seed42 pools',
}


In [ ]:
# Код evaluator встроен в ноутбук: клонировать репозиторий не требуется.
MODULES = {'worker.py': '"""CPU-only bounded evaluator for LLM-authored pandas feature programs."""\nimport os\nif __name__==\'__main__\':\n    os.environ[\'CUDA_VISIBLE_DEVICES\']=\'\'\n    for key in [\'OMP_NUM_THREADS\',\'MKL_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'NUMEXPR_NUM_THREADS\']:\n        os.environ[key]=\'1\'\nimport argparse\nimport ast\nimport hashlib\nimport gc\nimport json\nfrom pathlib import Path\nimport resource\nimport signal\nimport time\nimport numpy as np\nimport pandas as pd\n\nRAW = \'\'\'def modify_features(X, history):\n    return pd.DataFrame(index=X.index)\n\'\'\'\nDISALLOWED_CALLS={\'open\',\'eval\',\'exec\',\'compile\',\'getattr\',\'setattr\',\'delattr\',\'globals\',\'locals\',\'vars\',\n                  \'input\',\'breakpoint\',\'help\',\'dir\',\'memoryview\',\'type\',\'super\',\'__import__\'}\nDISALLOWED_ATTRIBUTES={\'eval\',\'query\',\'read_csv\',\'read_parquet\',\'read_pickle\',\'read_json\',\'read_sql\',\n                        \'read_excel\',\'read_html\',\'read_feather\',\'load\',\'loads\',\'save\',\'savetxt\',\'savez\',\n                        \'savez_compressed\',\'fromfile\',\'tofile\',\'loadtxt\',\'genfromtxt\',\'memmap\',\'ctypes\',\n                        \'f2py\',\'testing\',\'lib\',\'io\',\'compat\'}\n\n\ndef check_program(source):\n    if not isinstance(source,str) or len(source)>20000:\n        raise ValueError(\'Program must be <=20000 characters\')\n    tree=ast.parse(source)\n    functions=[n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name==\'modify_features\']\n    if len(functions)!=1 or [a.arg for a in functions[0].args.args]!=[\'X\',\'history\']:\n        raise ValueError(\'Define exactly def modify_features(X, history)\')\n    for node in ast.walk(tree):\n        if isinstance(node,(ast.Import,ast.ImportFrom,ast.ClassDef,ast.Global,ast.Nonlocal,ast.AsyncFunctionDef,ast.With)):\n            raise ValueError(\'No imports/classes/global state/context managers: pd and np are already supplied\')\n        if isinstance(node,ast.Name) and (node.id.startswith(\'_\') or node.id in DISALLOWED_CALLS):\n            raise ValueError(\'Forbidden name: \'+node.id)\n        if isinstance(node,ast.Attribute):\n            name=node.attr\n            if name.startswith(\'_\') or name.startswith((\'read_\',\'to_\')) and name not in {\'to_numpy\',\'to_frame\',\'to_dict\',\'to_list\',\'to_datetime\',\'to_numeric\'} or name in DISALLOWED_ATTRIBUTES:\n                raise ValueError(\'Forbidden attribute: \'+name)\n        if isinstance(node,ast.FunctionDef) and node.decorator_list:\n            raise ValueError(\'No decorators\')\n    for node in tree.body:\n        if not isinstance(node,(ast.FunctionDef,ast.Expr)) or isinstance(node,ast.Expr) and not isinstance(node.value,ast.Constant):\n            raise ValueError(\'Only function definitions at module level\')\n    return tree\n\n\ndef compile_program(source):\n    tree=check_program(source)\n    names=[\'abs\',\'min\',\'max\',\'sum\',\'len\',\'range\',\'enumerate\',\'zip\',\'list\',\'tuple\',\'dict\',\'set\',\'sorted\',\n           \'float\',\'int\',\'str\',\'bool\',\'round\',\'isinstance\',\'any\',\'all\',\'ValueError\',\'Exception\']\n    import builtins\n    scope={\'pd\':pd,\'np\':np,\'__builtins__\':{n:getattr(builtins,n) for n in names}}\n    exec(compile(tree,\'<llm-features>\',\'exec\'),scope)\n    return scope[\'modify_features\']\n\n\nclass Data:\n    def __init__(self,root):\n        self.root=Path(root)\n        self.manifest=json.loads((self.root/\'manifest.json\').read_text())\n        self.items=pd.read_parquet(self.root/\'articles.parquet\').set_index(\'article_id\')\n        # XGBoost allocates categorical histograms up to the maximum code. Compact\n        # all static item categories consistently in both X and history.\n        self.category_maps={}\n        for column in self.items:\n            encoded,categories=pd.factorize(self.items[column],sort=True)\n            self.category_maps[column]=categories.tolist()\n            self.items[column]=encoded.astype(\'int32\')\n        self.users=pd.read_parquet(self.root/\'customers.parquet\').set_index(\'customer_id\')\n        self.tx=pd.read_parquet(self.root/\'history.parquet\')\n        self.raw_u=[\'raw_u_\'+c.lower() for c in self.users.columns]\n        self.raw_i=[\'raw_i_\'+c for c in self.items.columns]\n\n    def inputs(self,pairs,cutoff):\n        pairs=pairs[[\'customer_id\',\'article_id\']].reset_index(drop=True)\n        users=self.users.reindex(pairs.customer_id).reset_index(drop=True)\n        items=self.items.reindex(pairs.article_id).reset_index(drop=True)\n        users.columns=self.raw_u\n        items.columns=self.raw_i\n        X=pd.concat([pairs,users,items],axis=1)\n        cutoff=pd.Timestamp(cutoff)\n        history=self.tx[(self.tx.t_dat<=cutoff)&(self.tx.t_dat>cutoff-pd.Timedelta(days=180))].copy()\n        assert (history.t_dat<=cutoff).all()\n        history[\'days_ago\']=(cutoff-history.t_dat).dt.days.astype(\'int32\')\n        history=history.drop(columns=\'t_dat\').merge(self.items.reset_index(),on=\'article_id\',how=\'left\',validate=\'many_to_one\')\n        history=history.merge(self.users[[\'age\']].rename(columns={\'age\':\'customer_age\'}).reset_index(),on=\'customer_id\',how=\'left\',validate=\'many_to_one\')\n        return X,history\n\n\ndef matrix(data,pairs,func,expected=None):\n    parts=[]\n    schema=None\n    for cutoff,frame in pairs.groupby(\'timestamp\',sort=False):\n        X,history=data.inputs(frame,cutoff)\n        # Only safe columns, no labels/targets/timestamps, are passed to the program.\n        def expired(*_):\n            raise TimeoutError(\'Feature function exceeded30 seconds for one snapshot\')\n        signal.signal(signal.SIGALRM,expired)\n        signal.alarm(30)\n        try:\n            out=func(X.copy(deep=True),history.copy(deep=True))\n        finally:\n            signal.alarm(0)\n        if not isinstance(out,pd.DataFrame) or not out.index.equals(X.index):\n            raise ValueError(\'Return a DataFrame with unchanged X.index and one row per pair\')\n        if len(out.columns)>48 or out.columns.duplicated().any():\n            raise ValueError(\'At most48 unique generated feature columns\')\n        for col in out.columns:\n            if not isinstance(col,str) or not col.startswith(\'f_\') or not pd.api.types.is_numeric_dtype(out[col]):\n                raise ValueError(\'Generated features must be numeric with f_ names; do not return IDs/raw columns\')\n            if np.array_equal(out[col].to_numpy(),X.customer_id.to_numpy()) or np.array_equal(out[col].to_numpy(),X.article_id.to_numpy()):\n                raise ValueError(\'IDs are join keys, not numeric features\')\n        raw=X[data.raw_u+data.raw_i]\n        full=pd.concat([raw,out],axis=1)\n        if schema is None:\n            schema=list(full)\n        if list(full)!=schema or expected is not None and schema!=expected:\n            raise ValueError(\'Feature schema changed between snapshots or splits\')\n        values=full.to_numpy(dtype=np.float32)\n        values[~np.isfinite(values)]=np.nan\n        parts.append(pd.DataFrame(values,index=frame.index,columns=schema))\n    values=pd.concat(parts).reindex(pairs.index).to_numpy(dtype=np.float32)\n    return values,schema\n\n\ndef map_at_12(predictions,targets):\n    if len(predictions)!=len(targets):\n        raise ValueError(\'User alignment changed\')\n    hits=np.array([np.isin(p,t) for p,t in zip(predictions,targets)])\n    denominator=np.minimum(12,np.array([len(np.unique(t)) for t in targets]))\n    if (denominator==0).any():\n        raise ValueError(\'Empty target list\')\n    ap=((hits.cumsum(axis=1)/np.arange(1,13))*hits).sum(axis=1)/denominator\n    return float(ap.mean()),ap\n\n\ndef rank(scores,pairs,targets):\n    predictions=[]\n    if list(pairs.customer_id.drop_duplicates())!=list(targets.customer_id):\n        raise ValueError(\'Pool/user order changed\')\n    for _,frame in pairs.assign(score=scores).groupby(\'customer_id\',sort=False):\n        if len(frame)<12 or frame.article_id.duplicated().any():\n            raise ValueError(\'Invalid candidate pool\')\n        predictions.append(frame.sort_values([\'score\',\'article_id\'],ascending=[False,True]).article_id.head(12).to_numpy())\n    return np.stack(predictions)\n\n\ndef limits(memory_gb):\n    size=int(memory_gb*1024**3)\n    resource.setrlimit(resource.RLIMIT_AS,(size,size))\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',required=True)\n    parser.add_argument(\'--directory\',required=True)\n    parser.add_argument(\'--stage\',choices=[\'search\',\'confirm\',\'val\',\'test\',\'replay_val\',\'replay_test\'],required=True)\n    parser.add_argument(\'--threads\',type=int,default=4)\n    parser.add_argument(\'--memory-gb\',type=float,default=4)\n    parser.add_argument(\'--backend\',choices=[\'xgboost\',\'lightgbm\'],default=\'lightgbm\')\n    args=parser.parse_args()\n    limits(args.memory_gb)\n    tick=time.perf_counter()\n    destination=Path(args.directory)\n    source=(destination/\'program.py\').read_text()\n    func=compile_program(source)\n    data=Data(args.data)\n    model_path=destination/(\'model.json\' if args.backend==\'xgboost\' else \'model.txt\')\n    if args.stage==\'search\' and not model_path.exists():\n        pairs=pd.read_parquet(data.root/\'train_pairs.parquet\')\n        values,names=matrix(data,pairs,func)\n        if source!=RAW and len(names)==len(data.raw_u+data.raw_i):\n            raise ValueError(\'Program generated no new features\')\n        categorical=set(data.raw_i+[\'raw_u_club_member_status\',\'raw_u_fashion_news_frequency\'])\n        kinds=[\'c\' if n in categorical else \'q\' for n in names]\n        for i,kind in enumerate(kinds):\n            if kind==\'c\':\n                values[values[:,i]<0,i]=np.nan\n        history={}\n        if args.backend==\'xgboost\':\n            import xgboost as xgb\n            dataset=xgb.QuantileDMatrix(values,label=pairs.label.values,feature_names=names,feature_types=kinds,\n                                       enable_categorical=True,nthread=args.threads,max_bin=127)\n            params={\'objective\':\'binary:logistic\',\'eval_metric\':\'logloss\',\'tree_method\':\'hist\',\'device\':\'cpu\',\n                \'max_depth\':6,\'eta\':.05,\'subsample\':.8,\'colsample_bytree\':.9,\'max_bin\':127,\n                \'seed\':42,\'nthread\':args.threads,\'max_cached_hist_node\':64}\n            model=xgb.train(params,dataset,num_boost_round=150,evals=[(dataset,\'train\')],evals_result=history,verbose_eval=False)\n            model.save_model(model_path)\n        else:\n            import lightgbm as lgb\n            params={\'objective\':\'binary\',\'metric\':\'binary_logloss\',\'verbosity\':-1,\'num_threads\':args.threads,\n                    \'learning_rate\':.05,\'num_leaves\':31,\'min_data_in_leaf\':100,\'max_bin\':127,\n                    \'feature_fraction\':.9,\'bagging_fraction\':.8,\'bagging_freq\':1,\'deterministic\':True,\n                    \'force_col_wise\':True,\'seed\':42,\'feature_fraction_seed\':42,\'bagging_seed\':42,\'data_random_seed\':42}\n            dataset=lgb.Dataset(values,label=pairs.label.values,feature_name=names,categorical_feature=[n for n in names if n in categorical])\n            model=lgb.train(params,dataset,num_boost_round=150,valid_sets=[dataset],valid_names=[\'train\'],callbacks=[lgb.record_evaluation(history)])\n            history={\'train\':{\'logloss\':history[\'train\'][\'binary_logloss\']}}\n            model.save_model(str(model_path))\n        (destination/\'schema.json\').write_text(json.dumps({\'features\':names,\'types\':kinds,\'params\':params,\'train_pairs\':len(pairs),\n                                                          \'item_category_maps\':data.category_maps}))\n        (destination/\'learning_curve.json\').write_text(json.dumps(history))\n        del pairs,values,dataset,model\n        gc.collect()\n    if args.backend==\'xgboost\':\n        import xgboost as xgb\n        model=xgb.Booster()\n        model.load_model(model_path)\n        model.set_param({\'nthread\':args.threads,\'device\':\'cpu\'})\n    else:\n        import lightgbm as lgb\n        model=lgb.Booster(model_file=str(model_path))\n    schema=json.loads((destination/\'schema.json\').read_text())\n    split=args.stage.replace(\'replay_\',\'\')\n    pairs=pd.read_parquet(data.root/f\'{split}_pairs.parquet\')\n    targets=pd.read_parquet(data.root/f\'{split}_targets.parquet\')\n    values,names=matrix(data,pairs,func,expected=schema[\'features\'])\n    for i,kind in enumerate(schema[\'types\']):\n        if kind==\'c\':\n            values[values[:,i]<0,i]=np.nan\n    if args.backend==\'xgboost\':\n        scores=model.predict(xgb.DMatrix(values,feature_names=names,feature_types=schema[\'types\'],enable_categorical=True,nthread=args.threads))\n    else:\n        scores=model.predict(values,num_threads=args.threads)\n    predictions=rank(scores,pairs,targets)\n    score,ap=map_at_12(predictions,targets.article_id)\n    if args.stage.startswith(\'replay_\'):\n        assert np.array_equal(predictions,np.load(destination/f\'{split}_predictions.npy\'))\n        assert np.array_equal(ap,np.load(destination/f\'{split}_user_ap.npy\'))\n    else:\n        np.save(destination/f\'{split}_predictions.npy\',predictions)\n        np.save(destination/f\'{split}_user_ap.npy\',ap)\n    metrics={\'map\':score,\'users\':len(targets),\'pairs\':len(pairs),\'seconds\':time.perf_counter()-tick,\n             \'program_sha256\':hashlib.sha256(source.encode()).hexdigest(),\n             \'model_sha256\':hashlib.sha256(model_path.read_bytes()).hexdigest(),\'backend\':args.backend,\n             \'stage\':args.stage,\'threads\':args.threads,\'passed\':True,\'features\':names}\n    (destination/f\'{args.stage}_metrics.json\').write_text(json.dumps(metrics,indent=2))\n    print(json.dumps(metrics))\n\n\nif __name__==\'__main__\':\n    main()\n', 'runtime.py': '"""Notebook host: paper-style islands, local generation callback and CPU evaluator."""\nimport ast\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport time\nimport zipfile\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom worker import RAW, check_program, Data, map_at_12\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef write_json(path,obj):\n    path=Path(path)\n    temp=path.with_suffix(path.suffix+\'.tmp\')\n    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef locate_bundle(input_root,working,expected_manifest_sha):\n    input_root=Path(input_root)\n    matches=[]\n    for path in input_root.rglob(\'manifest.json\'):\n        try:\n            if json.loads(path.read_text()).get(\'bundle_id\')==\'hm_llmfe_kaggle_fulltrain_v2\':\n                matches.append(path.parent)\n        except (ValueError,OSError):\n            pass\n    if not matches:\n        archives=list(input_root.rglob(\'hm_llmfe_kaggle_fulltrain_v2.zip\'))\n        if len(archives)!=1:\n            raise FileNotFoundError(\'Add the supplied hm_llmfe_kaggle_fulltrain_v2.zip as a Kaggle Dataset/Input\')\n        destination=Path(working)/\'hm_llmfe_input\'\n        destination.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for member in z.infolist():\n                if not (destination/member.filename).resolve().is_relative_to(destination.resolve()):\n                    raise ValueError(\'Archive path outside destination\')\n            z.extractall(destination)\n        matches=[destination]\n    if len(matches)!=1:\n        raise ValueError(\'Several H&M bundles attached; keep exactly one\')\n    root=matches[0]\n    if sha(root/\'manifest.json\')!=expected_manifest_sha:\n        raise ValueError(\'Unexpected bundle manifest/version\')\n    manifest=json.loads((root/\'manifest.json\').read_text())\n    for name,info in manifest[\'files\'].items():\n        if sha(root/name)!=info[\'sha256\']:\n            raise ValueError(\'Data checksum mismatch: \'+name)\n    return root,manifest\n\n\ndef evaluate(data,directory,stage,config):\n    directory=Path(directory)\n    backend=config.get(\'backend\',config.get(\'search_backend\',\'lightgbm\'))\n    result=directory/f\'{stage}_metrics.json\'\n    if result.exists():\n        metrics=json.loads(result.read_text())\n        if metrics.get(\'backend\')!=backend:\n            raise RuntimeError(\'Cached result uses another evaluator; start a new run ID\')\n        model_path=directory/(\'model.txt\' if metrics.get(\'backend\')==\'lightgbm\' else \'model.json\')\n        if metrics[\'program_sha256\']!=sha(directory/\'program.py\') or metrics[\'model_sha256\']!=sha(model_path):\n            raise RuntimeError(\'Cached result does not match model/program\')\n        return metrics\n    command=[sys.executable,str(Path(__file__).with_name(\'worker.py\')),\'--data\',str(data),\n             \'--directory\',str(directory),\'--stage\',stage,\'--threads\',str(1 if stage.startswith(\'replay_\') else config[\'cpu_threads\']),\n             \'--memory-gb\',str(config[\'evaluation_memory_gb\']),\'--backend\',backend]\n    env={k:v for k,v in os.environ.items() if not any(s in k.upper() for s in [\'TOKEN\',\'SECRET\',\'API_KEY\'])}\n    env.update(CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',NUMEXPR_NUM_THREADS=\'1\')\n    with (directory/f\'{stage}.log\').open(\'w\') as log:\n        try:\n            process=subprocess.run(command,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=config[\'evaluation_timeout_seconds\'])\n        except subprocess.TimeoutExpired:\n            raise TimeoutError(f"Evaluation exceeded {config[\'evaluation_timeout_seconds\']} seconds") from None\n    if process.returncode or not result.exists():\n        raise RuntimeError((directory/f\'{stage}.log\').read_text()[-2000:])\n    return json.loads(result.read_text())\n\n\ndef source_from_response(text):\n    blocks=re.findall(r\'```(?:python)?\\s*\\n(.*?)```\',text,re.S|re.I)\n    source=next((b for b in blocks if \'def modify_features\' in b),text).strip()+\'\\n\'\n    check_program(source)\n    return source\n\n\ndef source_key(source):\n    tree=ast.parse(source)\n    for node in ast.walk(tree):\n        if isinstance(node,ast.FunctionDef) and node.body and isinstance(node.body[0],ast.Expr) and isinstance(node.body[0].value,ast.Constant) and isinstance(node.body[0].value.value,str):\n            node.body=node.body[1:]\n    return hashlib.sha256(ast.dump(tree,include_attributes=False).encode()).hexdigest()\n\n\ndef choose_demonstrations(memory,rng):\n    clusters={}\n    for item in memory:\n        clusters.setdefault(item[\'score\'],[]).append(item)\n    scores=np.array(list(clusters),dtype=float)\n    temperature=.1*(1-(len(memory)%10000)/10000)\n    weights=np.exp((scores-scores.max())/temperature)\n    weights/=weights.sum()\n    count=min(3,len(clusters))\n    selected=rng.choice(len(scores),size=count,replace=False,p=weights)\n    return [clusters[scores[i]][int(rng.integers(len(clusters[scores[i]])))] for i in selected]\n\n\ndef prompt(data,memory,seed):\n    demo=choose_demonstrations(memory,np.random.default_rng(seed))\n    dataset=Data(data)\n    pairs=pd.read_parquet(Path(data)/\'train_pairs.parquet\')\n    selected=pairs.sort_values(\'timestamp\').tail(200).sample(5,random_state=42)\n    X,_=dataset.inputs(selected,selected.timestamp.iloc[0])\n    examples=X.drop(columns=[\'customer_id\',\'article_id\']).assign(training_label=selected.label.to_numpy()).to_dict(\'records\')\n    descriptions={\n        \'task\':\'H&M: rank fixed candidate articles per customer for purchases in next7days; maximize validation MAP@12.\',\n        \'function\':\'def modify_features(X, history): return generated_features_dataframe\',\n        \'X_columns\':list(X.columns),\n        \'history_columns\':[\'customer_id\',\'article_id\',\'price\',\'sales_channel_id\',\'days_ago\',\'customer_age\',*dataset.manifest[\'item_columns\']],\n        \'history_scope\':\'time-safe past180 days, selected customer cohort; not population-wide sales history\',\n        \'examples\':examples}\n    instructions=\'\'\'Design useful nontrivial recommendation features using clothing/customer domain knowledge.\nReturn ONE complete Python program in a python code block, defining modify_features(X, history).\npd (pandas) and np (numpy) are supplied: NO imports, file/network access or other libraries.\nX has one row per candidate pair. history contains only past transactions; no future data/labels.\nUse customer_id/article_id only as join/grouping keys, never as numeric predictions.\nGenerate up to48 numeric columns named f_*. Return a DataFrame with exactly X.index and row order.\nReturn ONLY new features; the host always appends the same15 raw static features.\nPrefer vectorized groupby/merge/map, time windows, affinity and interactions. Missing values allowed.\nHistory and IDs enable joins; all frame rows belong to ONE cutoff per invocation. No timestamps needed.\nYour program is called separately on every training snapshot and evaluation split; its schema must stay fixed.\nNo fitting on validation/test. No globals, classes, decorators, OS calls, randomness or rowwise Python loops.\nThe evaluator trains CPU LightGBM150 rounds/31 leaves, scores fixed candidates and computes MAP@12 on256 validation users.\nEach call of your function must finish within30 seconds. The full803377-pair training\nevaluation has a separate overall timeout set by the host, recorded in the run configuration.\nThe entire program must fit2048 output tokens. Include a short hypothesis as a function docstring.\nFive training examples, schema and feature descriptions follow:\n\'\'\'\n    metadata=\'\'\'raw_u_age=age; raw_u_fn/active=customer account flags; membership/news are encoded categories.\nraw_i_* describe clothing product family/type, graphic appearance, colour, department/index/section/garment group.\nprice=normalized transaction price; days_ago=integer days before cutoff; sales_channel_id=channel1/2.\nEvery proposal must specify its COMPLETE chosen feature set, not a patch to previous code.\n\'\'\'\n    prior=[{\'validation_map_fraction\':t[\'score\'],\'program\':(Path(t[\'directory\'])/\'program.py\').read_text()} for t in demo]\n    return instructions+metadata+json.dumps(descriptions,separators=(\',\',\':\'),default=str)+\'\\nPrior programs:\\n\'+json.dumps(prior,separators=(\',\',\':\'))\n\n\ndef progress(out,trials):\n    out=Path(out)\n    write_json(out/\'trials.json\',trials)\n    valid=[t for t in trials if t[\'status\']==\'valid\']\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    if valid:\n        x=[t[\'sample\'] for t in valid]\n        y=100*np.array([t[\'score\'] for t in valid])\n        axes[0].scatter(x,y,label=\'Candidate\')\n        axes[0].step(x,np.maximum.accumulate(y),where=\'post\',label=\'Best search\')\n        axes[0].legend()\n        axes[1].hist(y,bins=min(10,len(valid)))\n    axes[0].set(xlabel=\'Generated program\',ylabel=\'Search MAP@12 (%)\')\n    axes[1].set(xlabel=\'Search MAP@12 (%)\',ylabel=\'Evaluated programs\')\n    fig.suptitle(\'Search only: fixed256 users/candidates — not test\')\n    fig.tight_layout();fig.savefig(out/\'search_progress.png\',dpi=150);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    labels=[\'valid\',\'rejected\',\'duplicate\']\n    axes[0].bar(labels,[sum(t[\'status\']==s for t in trials) for s in labels])\n    axes[0].set_ylabel(\'Generated programs\')\n    for t in valid:\n        p=Path(t[\'directory\'])/\'learning_curve.json\'\n        history=json.loads(p.read_text())[\'train\'][\'logloss\']\n        axes[1].plot(np.arange(1,len(history)+1),history,label=str(t[\'sample\']),alpha=.7)\n    axes[1].set(xlabel=\'Boosting round\',ylabel=\'Train binary logloss\')\n    if valid:\n        axes[1].legend(ncol=4,fontsize=7)\n    fig.tight_layout();fig.savefig(out/\'training_and_rejections.png\',dpi=150);plt.close(fig)\n    lines=[\'# Search progress\',\'\',f\'Generated: {len(trials)}; evaluated distinct: {len(valid)}.\',\n           \'Search validation only; no final test results yet.\',\'\',\n           \'| Sample | Island | Status | MAP@12, % |\',\'|---|---:|---|---:|\']\n    for t in trials:\n        value=f"{100*t[\'score\']:.5f}" if t[\'status\']==\'valid\' else \'—\'\n        lines.append(f"| {t[\'sample\']} | {t[\'island\']} | {t[\'status\']} | {value} |")\n    lines+=[\'\',\'![Search](search_progress.png)\',\'![Training](training_and_rejections.png)\']\n    (out/\'progress.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n\n\ndef search(data,out,config,generate):\n    data,out=Path(data),Path(out)\n    out.mkdir(parents=True,exist_ok=True)\n    if (out/\'selected.json\').exists():\n        return json.loads((out/\'selected.json\').read_text())\n    control=out/\'raw_control\';control.mkdir(exist_ok=True)\n    (control/\'program.py\').write_text(RAW)\n    baseline=evaluate(data,control,\'search\',config)\n    raw={\'directory\':str(control),\'score\':baseline[\'map\']}\n    memories=[[raw.copy()] for _ in range(3)]\n    trials=[]\n    seen=set()\n    samples=config[\'llm_samples\']\n    for index in range(samples):\n        batch=index//3\n        batchdir=out/f\'batch_{batch:02d}\';batchdir.mkdir(exist_ok=True)\n        if not (batchdir/\'meta.json\').exists():\n            island=int(np.random.default_rng(42+batch).integers(3))\n            text=prompt(data,memories[island],42+batch)\n            (batchdir/\'prompt.txt\').write_text(text)\n            write_json(batchdir/\'meta.json\',{\'island\':island,\'programs\':min(3,samples-batch*3),\'seed\':42+batch})\n        meta=json.loads((batchdir/\'meta.json\').read_text());island=meta[\'island\']\n        directory=out/\'candidates\'/f\'candidate_{index+1:03d}\';directory.mkdir(parents=True,exist_ok=True)\n        if (directory/\'trial.json\').exists():\n            trial=json.loads((directory/\'trial.json\').read_text())\n        else:\n            trial={\'sample\':index+1,\'island\':island,\'batch\':batch,\'directory\':str(directory)}\n            tick=time.perf_counter()\n            try:\n                response_path=directory/\'response.json\'\n                if response_path.exists():\n                    response=json.loads(response_path.read_text())\n                else:\n                    response=generate((batchdir/\'prompt.txt\').read_text(),42+index)\n                    write_json(response_path,response)\n                if response.get(\'truncated\'):\n                    raise ValueError(\'Program truncated at output token limit\')\n                code=source_from_response(response[\'text\'])\n                (directory/\'program.py\').write_text(code)\n                key=source_key(code);trial[\'program_key\']=key\n                if key in seen:\n                    trial.update(status=\'duplicate\',error=\'Duplicate Python AST\')\n                else:\n                    metrics=evaluate(data,directory,\'search\',config)\n                    trial.update(status=\'valid\',score=metrics[\'map\'],metrics=metrics)\n            except Exception as error:\n                trial.update(status=\'rejected\',error=str(error))\n            trial[\'elapsed_seconds\']=time.perf_counter()-tick\n            write_json(directory/\'trial.json\',trial)\n        trials.append(trial)\n        if trial[\'status\']==\'valid\':\n            seen.add(trial[\'program_key\'])\n            if trial[\'score\']>max(t[\'score\'] for t in memories[island]):\n                memories[island].append(trial)\n        write_json(out/\'memory.json\',memories)\n        progress(out,trials)\n        print(f"sample{index+1}/{samples}, island{island}: {trial[\'status\']}; measured={len(seen)}",flush=True)\n    # Raw control can win; selecting a harmful LLM program is not forced.\n    options=[raw]+[t for t in trials if t[\'status\']==\'valid\']\n    options=sorted(options,key=lambda t:t[\'score\'],reverse=True)[:3]\n    confirmations=[]\n    for t in options:\n        metrics=evaluate(data,Path(t[\'directory\']),\'confirm\',config)\n        confirmations.append({\'directory\':t[\'directory\'],\'score\':metrics[\'map\'],\'metrics\':metrics})\n        write_json(out/\'confirmation.json\',confirmations)\n    best=max(confirmations,key=lambda t:t[\'score\'])\n    selected={**best,\'selection\':\'512 disjoint validation users; frozen before heldout val/test\',\n              \'selected_raw_control\':Path(best[\'directory\'])==control}\n    write_json(out/\'selected.json\',selected)\n    return selected\n\n\ndef final(data,out,config):\n    data,out=Path(data),Path(out)\n    selected=json.loads((out/\'selected.json\').read_text())\n    choices={\'Raw LightGBM\':out/\'raw_control\',\'LLM-FE LightGBM\':Path(selected[\'directory\'])}\n    metrics={}\n    for name,directory in choices.items():\n        settings={**config,\'backend\':\'lightgbm\'}\n        metrics[name]={}\n        for split in [\'val\',\'test\']:\n            original=evaluate(data,directory,split,settings)\n            replay=evaluate(data,directory,\'replay_\'+split,settings)\n            assert original[\'map\']==replay[\'map\'] and replay[\'passed\']\n            metrics[name][split]=original\n    # Native reference is never included in prompts or model selection.\n    references={\'Manual features → LightGBM\':\'manual_lightgbm\',\n                \'GNN → LightGBM\':\'gnn_lightgbm\',\n                \'ID-GNN (graph training budget)\':\'idgnn\'}\n    for name,label in references.items():\n        metrics[name]={}\n        for split in [\'val\',\'test\']:\n            targets=pd.read_parquet(data/f\'{split}_targets.parquet\')\n            value,_=map_at_12(np.load(data/f\'{split}_{label}_reference.npy\'),targets.article_id)\n            metrics[name][split]={\'map\':value,\'reference\':True}\n    frame=pd.DataFrame([{\'model\':name,\'val_map\':r[\'val\'][\'map\'],\'test_map\':r[\'test\'][\'map\']} for name,r in metrics.items()])\n    frame.to_csv(out/\'comparison.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(10,4))\n    for i,split in enumerate([\'val\',\'test\']):\n        bars=ax.barh(np.arange(len(frame))+(i-.5)*.32,100*frame[split+\'_map\'],height=.32,label=split)\n        ax.bar_label(bars,fmt=\'%.3f\',padding=3,fontsize=8)\n    ax.set_yticks(np.arange(len(frame)),frame.model);ax.set_xlabel(\'MAP@12 (%) — identical fixed candidate pools\');ax.legend()\n    ax.set_xlim(0,max(.1,float(frame[[\'val_map\',\'test_map\']].to_numpy().max())*118))\n    if config.get(\'smoke\'):\n        fig.suptitle(\'SYNTHETIC SMOKE — not Llama results\')\n    fig.tight_layout();fig.savefig(out/\'comparison.png\',dpi=150,bbox_inches=\'tight\');plt.close(fig)\n    aps={name:np.load(path/\'test_user_ap.npy\') for name,path in choices.items()}\n    truth=pd.read_parquet(data/\'test_targets.parquet\')\n    for name,label in references.items():\n        aps[name]=map_at_12(np.load(data/f\'test_{label}_reference.npy\'),truth.article_id)[1]\n    pd.DataFrame({\'customer_id\':truth.customer_id.values,**aps}).to_csv(out/\'test_user_ap.csv\',index=False)\n    fig,axes=plt.subplots(1,2,figsize=(13,4))\n    for name,values in aps.items():\n        axes[0].hist(100*values[values>0],bins=np.linspace(0,100,21),histtype=\'step\',label=name)\n    axes[0].set(xlabel=\'Positive user AP@12 (%)\',ylabel=\'Users\',yscale=\'log\');axes[0].legend(fontsize=7)\n    axes[1].barh(list(aps),[100*np.mean(a==0) for a in aps.values()]);axes[1].set(xlabel=\'Test users without hits (%)\',xlim=(0,100))\n    if config.get(\'smoke\'):\n        fig.suptitle(\'SYNTHETIC SMOKE — not Llama results\')\n    fig.tight_layout();fig.savefig(out/\'ap_histograms.png\',dpi=150,bbox_inches=\'tight\');plt.close(fig)\n    delta=aps[\'LLM-FE LightGBM\']-aps[\'Raw LightGBM\']\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    axes[0].bar([\'LLM better\',\'Tie\',\'Raw better\'],[(delta>0).sum(),(delta==0).sum(),(delta<0).sum()])\n    if np.any(delta!=0):\n        axes[1].hist(100*delta[delta!=0],bins=30);axes[1].set_yscale(\'log\')\n    axes[1].set(xlabel=\'LLM minus raw AP (percentage points)\',ylabel=\'Users; ties omitted\')\n    fig.tight_layout();fig.savefig(out/\'paired_comparison.png\',dpi=150);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    for name in [\'Raw LightGBM\',\'LLM-FE LightGBM\']:\n        curve=json.loads((choices[name]/\'learning_curve.json\').read_text())[\'train\'][\'logloss\']\n        axes[0].plot(np.arange(1,len(curve)+1),curve,label=name)\n    axes[0].set(xlabel=\'Boosting round\',ylabel=\'Train binary logloss\');axes[0].legend()\n    delta_lgb=aps[\'LLM-FE LightGBM\']-aps[\'Raw LightGBM\']\n    axes[1].bar([\'LLM better\',\'Tie\',\'Raw better\'],[(delta_lgb>0).sum(),(delta_lgb==0).sum(),(delta_lgb<0).sum()])\n    axes[1].set_title(\'LightGBM: paired test AP\')\n    fig.tight_layout();fig.savefig(out/\'lightgbm_comparison.png\',dpi=150);plt.close(fig)\n    trials=json.loads((out/\'trials.json\').read_text())\n    report={\'complete\':True,\'metrics\':metrics,\'selected\':selected,\'config\':config,\n            \'smoke\':bool(config.get(\'smoke\',False)),\n            \'generated_programs\':len(trials),\'evaluated_distinct\':sum(t[\'status\']==\'valid\' for t in trials),\n            \'independent_full_replay\':True,\'coverage\':\'bounded exported subset, not entire H&M\',\n            \'paper_reproduction\':False}\n    write_json(out/\'report.json\',report)\n    lines=[\'# Llama3.1-8B → Python features → LightGBM: bounded H&M\',\'\',\n           \'Adaptation of LLM-FE; not a numerical reproduction of the paper. All MAP values in percent.\',\n           \'One seed42,180-day selected-cohort history, frozen shared ID-GNN pools.\',\n           \'LightGBM selects features throughout search and confirmation; all new trees use identical803377 training pairs.\',\n           \'Manual and GNN→LightGBM references use those same803377 supervised pairs and these exact evaluation rows/pools.\',\n           \'Tuning protocols, derived seeds and historical representations differ; GNN also used upstream graph training.\',\n           \'Compare pipeline quality on this subset, not differences in isolated feature quality or full-dataset MAP.\',\n           \'Selection uses only search256 + confirmation512; heldout val1024 and test2048 used after freezing.\',\n           \'Native ID-GNN reference used more training data and is contextual, not a matched-budget comparison.\',\n           f"Generated{len(trials)} programs; evaluated distinct{report[\'evaluated_distinct\']}. Raw control selected: {selected[\'selected_raw_control\']}.",\n           \'All raw/LLM heldout predictions reproduced from saved models in separate CPU processes with1 thread.\',\n           \'\', \'| Model | Validation MAP, % | Test MAP, % |\',\'|---|---:|---:|\']\n    if config.get(\'smoke\'):\n        lines=[\'# SYNTHETIC SMOKE — not Llama results\',\'\',\n               \'This report tests software using a synthetic generator and tiny data. No real Llama inference or quality experiment.\', \'\']+lines\n    lines += [f"| {r.model} | {100*r.val_map:.5f} | {100*r.test_map:.5f} |" for r in frame.itertuples()]\n    lines += [\'\',f"Selected program: `{Path(selected[\'directory\']).relative_to(out)}/program.py`.",\n              \'![MAP](comparison.png)\',\'![AP](ap_histograms.png)\',\'![Paired](paired_comparison.png)\',\n              \'![Search](search_progress.png)\',\'![Training](training_and_rejections.png)\',\n              \'![LightGBM](lightgbm_comparison.png)\']\n    (out/\'report.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    return report\n'}
CODE = WORK / 'llmfe_runtime'
CODE.mkdir(exist_ok=True)
for name, source in MODULES.items():
    (CODE / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE))
import importlib
import worker as feature_worker
importlib.reload(feature_worker)
import runtime as experiment
importlib.reload(experiment)


In [ ]:
DATA, manifest = experiment.locate_bundle(INPUT_ROOT, WORK, '3d042b7fb24e702cb0784a99c6f04f82774e70dc802f710744232cddf016372a')
assert manifest['train_pairs'] == 803377 and manifest['train_user_times'] == 50000
assert manifest['files']['train_pairs.parquet']['sha256'] == manifest['train_source_sha256']
CONFIG['bundle_manifest_sha256'] = '3d042b7fb24e702cb0784a99c6f04f82774e70dc802f710744232cddf016372a'
display(pd.DataFrame([{'split':k,'users':v['users'],'pairs':v['pairs']} for k,v in manifest['splits'].items()]))
print('Training pairs:', manifest['train_pairs'], 'history rows:', manifest['history_rows'])
print('Data checksums: OK')


## Доступ к Llama и фиксация запуска

`unsloth` скачивается анонимно, без чтения Kaggle Secrets и сохранённых HF-токенов.
Это полноразмерная публикация весов; NF4-квантизация выполняется при загрузке одинаково
для обоих источников. Unsloth-библиотека и её отдельное окружение не требуются.
`hf` требует одобрения Meta и `HF_TOKEN`. При403 проверьте доступ, при401 — токен.
Автоматического переключения источника нет. Ревизия публикации фиксируется отдельно;
побитовое совпадение весов двух издателей здесь не проверено. Лицензия Llama сохраняется.
Конфигурация, ревизия модели, версии пакетов, хеши данных и evaluator определяют новый run ID;
старые результаты не перезаписываются. Токен не сохраняется в артефакты и не выводится.


In [ ]:
import torch
from huggingface_hub import HfApi, hf_hub_download
if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU T4 в настройках Kaggle')
gpu = torch.cuda.get_device_properties(0)
if gpu.total_memory < 14 * 1024**3:
    raise RuntimeError('Нужно не менее14GiB GPU RAM; выбранная карта не подходит для этой конфигурации')

def load_kaggle_secret():
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret('HF_TOKEN')

token = resolve_auth(MODEL_SOURCE, os.environ, load_kaggle_secret)
try:
    revision = HfApi(token=token).model_info(MODEL_ID, revision=MODEL_REVISION or 'main').sha
    hf_hub_download(MODEL_ID, 'config.json', revision=revision, token=token)
except Exception:
    hint = 'Internet и существование ревизии' if MODEL_SOURCE == 'unsloth' else 'Internet, HF_TOKEN и одобрение доступа Meta'
    raise RuntimeError('Не удалось прочитать ' + MODEL_ID + '. Проверьте ' + hint) from None

CONFIG['model_revision'] = revision
CONFIG['gpu'] = gpu.name
CONFIG['versions'] = {p:importlib.metadata.version(p) for p in
    ['torch','transformers','accelerate','bitsandbytes','lightgbm','pandas','numpy','pyarrow']}
CONFIG['runtime_sha256'] = {name:hashlib.sha256(source.encode()).hexdigest() for name,source in MODULES.items()}
fingerprint = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
RUN_ID = CONFIG['protocol'] + '_' + fingerprint
OUT = WORK / RUN_ID
OUT.mkdir(exist_ok=True)
config_path = OUT / 'run_config.json'
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise RuntimeError('Config mismatch: start a new run directory')
experiment.write_json(config_path, CONFIG)
print('Run:', RUN_ID, '| source:', MODEL_SOURCE, '| model:', MODEL_ID, '| GPU:', gpu.name, '| revision:', revision)


In [ ]:
# Контроль без LLM-признаков: реальное обучение на всех803377 парах.
control = OUT / 'raw_control'
control.mkdir(exist_ok=True)
(control / 'program.py').write_text(experiment.RAW, encoding='utf-8')
baseline = experiment.evaluate(DATA, control, 'search', CONFIG)
print('Raw LightGBM search MAP@12:', 100 * baseline['map'], '%')
print('Control training/evaluation seconds:', baseline['seconds'])


## Загрузка Llama на GPU0

Это инференс, без fine-tuning. NF4 позволяет разместить веса и KV-кеш на одной T4.
Используется float16; BF16 и FlashAttention2 не требуются. CPU-деревья обучаются
в отдельном процессе и не занимают VRAM. Первое скачивание весов может быть долгим.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
set_seed(CONFIG['seed'])
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, token=token, trust_remote_code=False)
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, token=token,
    quantization_config=quantization, torch_dtype=torch.float16, device_map={'':0},
    attn_implementation='sdpa', trust_remote_code=False, low_cpu_mem_usage=True)
model.eval()
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
print('Model GPU footprint GiB:', model.get_memory_footprint()/1024**3)
experiment.write_json(OUT / 'model_runtime.json', {
    'model':MODEL_ID, 'model_source':MODEL_SOURCE, 'revision':revision, 'is_loaded_in_4bit':bool(model.is_loaded_in_4bit),
    'gpu':gpu.name, 'footprint_bytes':int(model.get_memory_footprint()),
    'allocated_vram_bytes':int(torch.cuda.memory_allocated(0))})


In [ ]:
def generate_program(prompt, seed):
    set_seed(seed)
    messages = [
        {'role':'system','content':'You are a feature engineering researcher. Follow the Python interface exactly.'},
        {'role':'user','content':prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda:0')
    prompt_tokens = inputs['input_ids'].shape[1]
    if prompt_tokens > CONFIG['max_input_tokens']:
        raise ValueError('Prompt exceeds input budget; no silent context truncation')
    endings = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')]
    endings = list({int(i) for i in endings if i is not None})
    tick = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
            do_sample=True, temperature=CONFIG['temperature'], top_p=CONFIG['top_p'], top_k=CONFIG['top_k'],
            num_return_sequences=1, eos_token_id=endings, pad_token_id=tokenizer.pad_token_id, use_cache=True)
    generated = output[0, prompt_tokens:]
    result = {'text':tokenizer.decode(generated, skip_special_tokens=True),
        'seed':seed, 'prompt_tokens':int(prompt_tokens), 'generated_tokens':int(len(generated)),
        'seconds':time.perf_counter()-tick,
        'truncated':len(generated)>=CONFIG['max_new_tokens'] and int(generated[-1]) not in endings}
    del output, inputs, generated
    return result

# b=3: одинаковый prompt, три независимых последовательных сэмпла; это экономит VRAM T4.
#20 — общий бюджет программ, включая ошибочные; это отдельный протокол от локального H002.
selected = experiment.search(DATA, OUT, CONFIG, generate_program)
print('Selection frozen before test:', selected)
display(Image(filename=str(OUT / 'search_progress.png')))
display(Image(filename=str(OUT / 'training_and_rejections.png')))


## Программа, выбранная LightGBM → итоговая оценка

Выбранные признаки фиксируются по confirmation LightGBM; test не влияет на выбор. Контроль без новых признаков
тоже может победить. Оба новых LightGBM обучаются на тех же803377 парах, что прежние
ручной и GNN→LightGBM. Все методы оцениваются на одинаковых val/test строках и пулах.
Старые модели не перетренируются: их сохранённые top12 включены в архив данных.
Протоколы подбора/производные seeds различаются, GNN обучалась дополнительно на графе.


In [ ]:
# Генерации больше не нужны; освобождаем VRAM.
del model
gc.collect()
torch.cuda.empty_cache()
report = experiment.final(DATA, OUT, CONFIG)
display(pd.read_csv(OUT / 'comparison.csv').assign(
    val_map=lambda x:100*x.val_map, test_map=lambda x:100*x.test_map))
print('Все новые модели: полный повтор предсказаний из сохранённых весов подтверждён.')


In [ ]:
for name in ['comparison.png','ap_histograms.png','paired_comparison.png','lightgbm_comparison.png']:
    display(Image(filename=str(OUT / name)))
display(Markdown((OUT / 'report.md').read_text()))


In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / (RUN_ID + '_results')), 'zip', root_dir=OUT)
display(FileLink(archive))
print('Скачайте ZIP из Output. Веса Llama и HF_TOKEN в него не включены.')


## Что смотреть после запуска

- `report.md`, `comparison.csv`: MAP на одной и той же небольшой выборке, включая три способа + контроли.
- `comparison.png`, `ap_histograms.png`, `paired_comparison.png`, `lightgbm_comparison.png`: сравнение и распределения.
- `search_progress.png`, `training_and_rejections.png`: поиск, отказы и train logloss; loss не выбирает победителя.
- `candidates/candidate_*/program.py`, `response.json`, `trial.json`: полные программы, ответы и причины отказов.
- `selected.json`, `confirmation.json`: проверяемая история выбора только по validation.
- `model.json`/`model.txt`, `schema.json`, `*_metrics.json`, `*_predictions.npy`: новые деревья и независимый повтор.
- `run_config.json`, `model_runtime.json`: версия Llama, квантизация, пакеты, хеши кода/данных.

Проверки AST и отдельный CPU-процесс ограничивают интерфейс, время и память, но не являются
полноценной изоляцией произвольного враждебного Python-кода. Generated code не получает токен,
метки или будущую историю через функцию. Новые LLM-признаки не заимствуются из ручных/GNN-признаков.
